# 数据结构与科学计算 · 第 02 讲
## 栈、队列、双端队列与二维网格搜索

> **延续第 01 讲**：你已经接触了 Python `list`、NumPy 数组、索引和时间复杂度。本讲研究一个不同的问题：**当数据不断到来时，应该以什么顺序处理？**

**本讲完成后，你将能够：**

1. 区分 **LIFO（后进先出）** 与 **FIFO（先进先出）**；
2. 用 Python `list` 实现栈，用 `collections.deque` 实现队列；
3. 理解为什么 `list.pop(0)` 通常比 `deque.popleft()` 慢；
4. 编写迭代版深度优先搜索（DFS）与广度优先搜索（BFS）；
5. 在二维结构化网格中用 BFS 计算最短网格路径与离散距离；
6. 认识 CPU 侧的数据结构与 NumPy/JAX 数组各自的分工。

**预计学习时间**：60–90 分钟讲解与实验，30–50 分钟练习。只需 Python 标准库；可选用 NumPy（第 01 讲已安装）。

## 课程位置与学习路线

| 讲次 | 内容 | 工程上的联系 |
|:--|:--|:--|
| 01（已完成） | 动态数组、NumPy、复杂度 | CFD 场变量、内存布局 |
| **02（本讲）** | **栈、队列、`deque`、DFS/BFS** | **网格连通性、任务调度** |
| 03 | 字典、集合与哈希表 | 缓存、快速查找 |
| 04 | 链表与动态数组的实现思想 | 内存与容器设计 |
| 05 | 树、堆与优先队列 | 自适应网格、优先任务 |
| 06 | 图与图算法 | 非结构化网格拓扑 |
| 07 | 稀疏矩阵及 COO/CSR | 离散 Poisson 方程 |
| 08 | 数据布局、性能与 JAX | GPU 数值计算与可微编程 |

> 不需要死记名称；持续追问：**某种存储与访问方式，为什么更适合某类任务？**

## 0. VS Code / Jupyter 环境

1. 在 VS Code 中打开此 `.ipynb` 文件，点击右上角 **Select Kernel**；
2. 选用前一讲使用的 Python 环境；
3. 每次按 `Shift + Enter` 运行当前单元；
4. 如果之前运行导致变量状态混乱，可使用 **Restart Kernel & Run All**；
5. 标记 `TODO` 的练习单元需要你自己修改。直接运行所有单元时，练习检测可能提示未完成，**不代表文件损坏**。

本讲核心内容只需要 Python 标准库，无须安装额外包。

In [1]:
import sys
from collections import deque
from time import perf_counter

print("Python:", sys.version.split()[0])
print("deque 已导入；可以开始第二讲。")

Python: 3.13.5
deque 已导入；可以开始第二讲。


---
## 1. 概念：抽象数据类型（ADT）与实际实现

你可能把“栈”“队列”想象成某一种具体的容器，但首先它们描述的是**允许哪些操作，以及处理元素的顺序**。

- **栈（Stack）**：`push` 加到顶部，`pop` 从顶部取出，**LIFO**（Last In, First Out）。
- **队列（Queue）**：`enqueue` 从尾部加入，`dequeue` 从头部取出，**FIFO**（First In, First Out）。
- **双端队列（Deque）**：允许从两端加入和移除；既能做栈，也能做队列。

简单比喻：栈像一摞盘子；队列像按先来后到排队；双端队列像可以从两端进出的一排格子。

**重要区分**：ADT 说的是行为契约，Python `list` / `deque` 则是可以实现这些行为的具体容器。不同实现，性能可能差很多。

| 操作 | 栈（Stack） | 队列（Queue） |
|:--|:--|:--|
| 插入 | 顶端 `push` | 尾部 `enqueue` |
| 移除 | 顶端 `pop` | 头部 `dequeue` |
| 下一个元素 | 最后加入的 | 最早加入的 |
| 常见算法 | 括号匹配、DFS、撤销 | BFS、调度、消息处理 |

## 2. 栈（Stack）：后进先出

依次压入 `A`、`B`、`C`：

```text
         顶部 top
          ┌───┐
          │ C │  ← 下一个取出的是 C
          ├───┤
          │ B │
          ├───┤
          │ A │
          └───┘
```

对于一般 Python 程序，使用 `list` 就能很好地实现栈：

- `stack.append(x)`：入栈，**均摊 $O(1)$**；
- `stack.pop()`：出栈，通常 **$O(1)$**；
- `stack[-1]`：读取栈顶，不删除，**$O(1)$**；
- `len(stack)`：元素个数，**$O(1)$**。

注意：`pop()` 作用于空列表会产生 `IndexError`。

In [2]:
stack = []
for name in ["A", "B", "C"]:
    stack.append(name)
    print(f"push {name}  -> {stack}")

print("栈顶但不取出：", stack[-1])
while stack:
    print("pop：", stack.pop(), "剩余：", stack)

push A  -> ['A']
push B  -> ['A', 'B']
push C  -> ['A', 'B', 'C']
栈顶但不取出： C
pop： C 剩余： ['A', 'B']
pop： B 剩余： ['A']
pop： A 剩余： []


### 2.1 从零封装一个栈：让接口表达意图

实际项目不一定需要自行编写 `Stack` 类，但封装有助于分清**操作语义**与**内部存储**。

In [3]:
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)

    def pop(self):
        if self.is_empty():
            raise IndexError("不能从空栈中弹出元素")
        return self._items.pop()

    def peek(self):
        if self.is_empty():
            raise IndexError("空栈没有栈顶")
        return self._items[-1]

    def is_empty(self):
        return len(self._items) == 0

    def __len__(self):
        return len(self._items)


s = Stack()
s.push(10)
s.push(20)
print("size:", len(s), "peek:", s.peek(), "pop:", s.pop())

size: 2 peek: 20 pop: 20


### 2.2 算法实例：括号匹配

我们希望验证 `([{}])` 合法，而 `([)]` 不合法。关键事实是：**最后打开的括号，必须最先闭合**，这正是栈的顺序。

伪代码：

1. 遇到左括号：压栈；
2. 遇到右括号：弹栈，检查是否对应；
3. 结束时栈必须为空。

这是一个线性时间算法：给定长度为 $n$ 的字符串，每个字符最多入栈或出栈一次，所以总时间为 $O(n)$；最坏情况下栈大小为 $O(n)$。

In [4]:
def balanced_brackets(text: str) -> bool:
    matching = {")": "(", "]": "[", "}": "{"}
    left = set(matching.values())
    stack = []

    for ch in text:
        if ch in left:
            stack.append(ch)
        elif ch in matching:
            if not stack or stack.pop() != matching[ch]:
                return False
    return len(stack) == 0


for sample in ["([{}])", "([)]", "a*(b+c)", "(()", "abc"]:
    print(f"{sample!r:12} -> {balanced_brackets(sample)}")

'([{}])'     -> True
'([)]'       -> False
'a*(b+c)'    -> True
'(()'        -> False
'abc'        -> True


## 3. 为什么搜索算法需要“栈”？先看 DFS

假设每个点可以连接到几个相邻点，我们想访问所有可达点。

**深度优先搜索（DFS, Depth-First Search）**：先沿着一条分支尽量深入，到尽头后返回寻找其他分支。显式栈可以保存“下一步还要处理的点”。

下面的小图以邻接表表示：

```text
A ── B ── D
│
└── C ── E
```

这是一个**图的例子**，我们在第 06 讲才会系统学习图结构。本讲先熟悉“栈如何控制处理顺序”。

In [5]:
graph = {
    "A": ["B", "C"],
    "B": ["A", "D"],
    "C": ["A", "E"],
    "D": ["B"],
    "E": ["C"],
}


def dfs_iterative(graph, start):
    stack = [start]
    visited = set()
    order = []

    while stack:
        node = stack.pop()  # 取出最后加入的候选点
        if node in visited:
            continue
        visited.add(node)
        order.append(node)

        # 逆序压栈，使图中较早列出的邻居优先访问。
        for neighbor in reversed(graph[node]):
            if neighbor not in visited:
                stack.append(neighbor)
    return order


print("DFS 访问顺序：", dfs_iterative(graph, "A"))

DFS 访问顺序： ['A', 'B', 'D', 'C', 'E']


> **理解两个容易混淆的容器**：`stack` 保存“待处理任务”；`visited` 保存“已经处理过的点”。后者使用集合 `set`，因为 `x in set` 平均时间近似 $O(1)$（会在第 03 讲展开）。这里可以先把 `set` 当作“高效判断有没有见过”的工具。

**思考**：如果把 `stack.pop()` 改成“最早加入的先取出”，访问顺序会发生什么变化？下一节给你答案。

---
## 4. 队列（Queue）：先进先出

```text
队头 head                              队尾 tail
   ↓                                      ↓
[  A  |  B  |  C  ]  ← 在右端加入 D
   ↑
下一次从左端取出 A
```

如果用列表实现：

```python
q = ["A", "B", "C"]
q.append("D")  # 尾部加入，均摊 O(1)
q.pop(0)      # 从头移除，需要移动其他元素，O(n)
```

问题出在 `pop(0)`：Python `list` 内部是动态数组，移除头部元素后，后面元素的引用需要整体前移。

队列最关心的是两端的操作，因此 Python 标准库提供了 `collections.deque`（读作 *deck*）。

In [6]:
from collections import deque

queue = deque(["A", "B", "C"])
queue.append("D")
print("入队 D：", list(queue))

first = queue.popleft()
print("出队：", first)
print("剩余：", list(queue))

入队 D： ['A', 'B', 'C', 'D']
出队： A
剩余： ['B', 'C', 'D']


### 4.1 双端队列（Deque）：四种主要操作

| 结构与操作 | 时间复杂度（通常/均摊） | 备注 |
|:--|:--|:--|
| `list.append(x)` | $O(1)$ 均摊 | 适合尾部加入 |
| `list.pop()` | $O(1)$ | 适合尾部删除 |
| `list.pop(0)` | $O(n)$ | **不推荐**作为频繁出队 |
| `deque.append(x)` | $O(1)$ | 右端入队 |
| `deque.appendleft(x)` | $O(1)$ | 左端插入 |
| `deque.pop()` | $O(1)$ | 右端取出 |
| `deque.popleft()` | $O(1)$ | 左端出队 |

`deque` 支持访问两端，但**不适合频繁按中间下标随机访问**；中间索引通常需要 $O(n)$ 时间。

上面是常用复杂度描述：`list.append` 的均摊常数时间并不表示所有单次操作的耗时都完全相等；实际测量还受到解释器、硬件与缓存影响。

In [7]:
d = deque([2, 3])
d.appendleft(1)
d.append(4)
print("两端加入：", list(d))
print("左端取出：", d.popleft())
print("右端取出：", d.pop())
print("最终：", list(d))

两端加入： [1, 2, 3, 4]
左端取出： 1
右端取出： 4
最终： [2, 3]


### 4.2 实验：`list.pop(0)` 与 `deque.popleft()`

下面的实验在**队列初始装有 $n$ 个元素，随后全部取出**的情况下，比较两种实现的耗时。一般而言，逐次列表头部删除的总移动量按 $n^2$ 增长；双端队列逐个取出的总工作量按 $n$ 增长。

测时数据受机器负载影响，**应主要关注规模增长趋势，不要把某次倍数当成固定结论**。

In [8]:
def drain_list(n):
    q = list(range(n))
    while q:
        q.pop(0)


def drain_deque(n):
    q = deque(range(n))
    while q:
        q.popleft()


def timed(fn, n):
    t0 = perf_counter()
    fn(n)
    return perf_counter() - t0


for n in (2_000, 8_000, 20_000):
    t_list = timed(drain_list, n)
    t_deque = timed(drain_deque, n)
    print(f"n={n:>6,} | list: {t_list:.5f}s | deque: {t_deque:.5f}s")

n= 2,000 | list: 0.00029s | deque: 0.00012s
n= 8,000 | list: 0.00383s | deque: 0.00029s
n=20,000 | list: 0.02098s | deque: 0.00060s


## 5. 使用队列实现 BFS：先近后远

**广度优先搜索（BFS, Breadth-First Search）**：从起点开始，先处理距离为 1 步的邻居，再处理距离为 2 步的点，依次向外推进。

BFS 的关键操作：

1. 把起点放入队列；
2. 从**队头**取出一个点；
3. 把未访问的邻居放到**队尾**；
4. 重复直到队列为空。

为什么使用 FIFO？因为这样能保证先处理已发现的较近层，再处理更远的层。

在**每条边权重相同**、每次跳转代价为 1 的图中，BFS 第一次到达某个点时得到的步数，就是它到起点的最短步数。带不同权重的最短路一般不能直接使用这个结论。

In [9]:
def bfs_graph(graph, start):
    q = deque([start])
    visited = {start}   # 入队时就标记，避免同一点反复入队
    distance = {start: 0}
    order = []

    while q:
        node = q.popleft()
        order.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                distance[neighbor] = distance[node] + 1
                q.append(neighbor)
    return order, distance


order, distance = bfs_graph(graph, "A")
print("BFS 访问顺序：", order)
print("距离 A 的最短边数：", distance)

BFS 访问顺序： ['A', 'B', 'C', 'D', 'E']
距离 A 的最短边数： {'A': 0, 'B': 1, 'C': 1, 'D': 2, 'E': 2}


### 5.1 DFS 与 BFS 的核心区别

| 比较 | DFS（栈） | BFS（队列） |
|:--|:--|:--|
| 下一步处理谁？ | 最后加入的待处理点 | 最早加入的待处理点 |
| 搜索特征 | 沿分支深入、再回溯 | 按层扩展、逐层向外 |
| 无权图最短路？ | 直接运行不保证最短 | **保证**最少边数 |
| 常见用途 | 连通区域、回溯、遍历 | 无权最短路、网格距离 |

图用邻接表存储时，常见 DFS / BFS 的时间复杂度都是

\[
O(V+E),
\]

其中 $V$ 是顶点数量，$E$ 是边数量。BFS 的队列和访问集合需要的额外空间通常为 $O(V)$；迭代 DFS 的额外空间也可以达到 $O(V)$。

下面把这个概念带到二维 CFD 网格。

---
## 6. 科学计算案例：二维网格上的 BFS

考虑二维**结构化网格**上的一个可通行区域：

- `.`：可通行的单元格；
- `#`：被遮挡（例如障碍物占据）的单元格；
- `(j, i)`：第 $j$ 行、第 $i$ 列，与第一讲的 NumPy 索引约定一致；
- 一个网格单元只与上下左右 **4 邻域**连接，不走对角线。

对应的邻居可以写成

\[
\mathcal{N}(j,i)=\{(j-1,i),(j+1,i),(j,i-1),(j,i+1)\},
\]

其中实际保留**在网格范围内且未被障碍物占据**的邻居。

在 CFD 程序里，这种遍历思路可用于**网格连通性检查**、**流体区域标记**等离散拓扑问题。

**特别注意**：这里计算的“步数距离”是离散网格图上的距离，**不是物理流线、流体传播速度，也不是连续 PDE 的数值解**。

In [10]:
# 每一行是字符串；列数必须相同。
mesh = [
    "..........",
    ".####...#.",
    ".#......#.",
    ".#..####..",
    ".#........",
    "...###....",
    "..........",
]

ny, nx = len(mesh), len(mesh[0])
assert all(len(row) == nx for row in mesh)
start, goal = (0, 0), (6, 9)
print(f"网格形状: ny={ny}, nx={nx}; 起点: {start}; 终点: {goal}")
for row in mesh:
    print(row)

网格形状: ny=7, nx=10; 起点: (0, 0); 终点: (6, 9)
..........
.####...#.
.#......#.
.#..####..
.#........
...###....
..........


### 6.1 用函数生成网格邻居

和第 01 讲的五点差分一样，四邻域仍然由 `(j, i)` 的上下左右构成。但五点差分读取邻居数值来计算导数；BFS 则把邻居当作**可以走到的下一个节点**。

这里不显式生成完整的图，而是**按需计算邻居**。对于规则结构化网格，这通常既直接又节省存储。

In [11]:
def neighbors4(grid, cell):
    j, i = cell
    ny, nx = len(grid), len(grid[0])
    for dj, di in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        jj, ii = j + dj, i + di
        if 0 <= jj < ny and 0 <= ii < nx and grid[jj][ii] != "#":
            yield (jj, ii)


print("起点的邻居：", list(neighbors4(mesh, start)))
print("一个障碍旁单元的邻居：", list(neighbors4(mesh, (2, 2))))

起点的邻居： [(1, 0), (0, 1)]
一个障碍旁单元的邻居： [(3, 2), (2, 3)]


### 6.2 计算起点到所有可达网格单元的最少步数

`distance[cell]` 保存起点到 `cell` 的最短步数；同时我们使用 `distance` 的键作为“是否已经访问”的判断，因此无需额外的 `visited` 集合。

**一个重要的实现细节**：在邻居**入队时**就记录距离，而非等到出队。这可以避免多个邻居重复把同一个点放入队列。

In [12]:
def bfs_distances(grid, start):
    ny, nx = len(grid), len(grid[0])
    sj, si = start
    if not (0 <= sj < ny and 0 <= si < nx):
        raise ValueError("起点超出网格范围")
    if grid[sj][si] == "#":
        raise ValueError("起点处于障碍物中")

    q = deque([start])
    distance = {start: 0}

    while q:
        cell = q.popleft()
        for nxt in neighbors4(grid, cell):
            if nxt not in distance:
                distance[nxt] = distance[cell] + 1
                q.append(nxt)
    return distance


dist = bfs_distances(mesh, start)
print("可到达单元格个数：", len(dist))
print("到终点的最少四邻域步数：", dist.get(goal, "无法到达"))

可到达单元格个数： 54
到终点的最少四邻域步数： 15


### 6.3 用文本把离散距离场可视化

下面的格式化输出不需要 Matplotlib：两位数字表示最少步数，`##` 表示障碍，`--` 表示不可达区域。

In [13]:
def print_distance_map(grid, distances):
    for j, row in enumerate(grid):
        cells = []
        for i, char in enumerate(row):
            if char == "#":
                cells.append("##")
            elif (j, i) in distances:
                cells.append(f"{distances[(j, i)]:02d}")
            else:
                cells.append("--")
        print(" ".join(cells))


print_distance_map(mesh, dist)

00 01 02 03 04 05 06 07 08 09
01 ## ## ## ## 06 07 08 ## 10
02 ## 10 09 08 07 08 09 ## 11
03 ## 09 10 ## ## ## ## 13 12
04 ## 08 09 10 11 12 13 14 13
05 06 07 ## ## ## 13 14 15 14
06 07 08 09 10 11 12 13 14 15


### 6.4 直观理解：二维网格中的复杂度

令网格尺寸为 $n_y\times n_x$，共 $N=n_y n_x$ 个单元。每个单元最多检查四个邻居：

\[
V\leq N,\qquad E\leq 2N\quad\text{（无向四邻域网格）}.
\]

因此，遍历整个网格的最坏情况下时间为

\[
O(V+E)=O(N)=O(n_y n_x),
\]

空间也可能达到 $O(N)$（队列与距离字典）。上式是渐近量级，不包含 Python 字典、元组及对象分配的实际常数开销。

**联系 CFD**：如果需要为整个二维流场频繁存储浮点量，NumPy/JAX 数组仍然更合适；但如果需要“按某个顺序逐步探索少量网格单元”，队列和集合就很自然。这里的目标不是用 `deque` 替换流场数组。

---
## 7. `deque(maxlen=...)`：有界历史与滑动窗口

`deque` 还有一种很有用的用法：**限定最大长度**。当队列已满，从右端 `append` 新元素会自动丢弃左端最旧的元素。

这适用于记录最近几次迭代的标量日志、传感器读数、诊断指标等。对强化学习中的大型 **Replay Buffer**，更常见的是预分配数组/张量或专门的缓冲区，因为采样和批量传输也很重要。

In [14]:
history = deque(maxlen=4)
for step, drag_coefficient in enumerate([1.2, 1.1, 1.05, 0.99, 0.96, 1.01], start=1):
    history.append(drag_coefficient)
    recent_mean = sum(history) / len(history)
    print(f"step={step}  history={list(history)}  recent mean={recent_mean:.3f}")

step=1  history=[1.2]  recent mean=1.200
step=2  history=[1.2, 1.1]  recent mean=1.150
step=3  history=[1.2, 1.1, 1.05]  recent mean=1.117
step=4  history=[1.2, 1.1, 1.05, 0.99]  recent mean=1.085
step=5  history=[1.1, 1.05, 0.99, 0.96]  recent mean=1.025
step=6  history=[1.05, 0.99, 0.96, 1.01]  recent mean=1.002


### 7.1 对 NumPy 与 JAX 的意义

一个科学计算程序里可能同时存在几类数据：

- **数值场**：速度、压力等大量浮点值 → NumPy/JAX `ndarray`；
- **图搜索的工作列表**：下一批待访问网格 → `deque`；
- **已经访问的离散坐标**：是否到达过某个网格 → `set` / `dict`；
- **最近若干步的性能指标**：残差、阻力系数 → 小型 `deque(maxlen=k)`。

> **JAX 注意**：Python 的 `deque` 是主机端的可变容器，不能把其动态 `append` / `popleft` 操作直接当成 `jax.jit` 内部的设备计算。对需要 JIT 编译的固定规模迭代，通常要使用数组以及 JAX 提供的控制流（例如 `jax.lax.scan`）；以后课程会单独介绍。

---
## 8. 动手练习（先做题，后看答案）

本讲设计了三个递进练习：

1. **栈**：自己实现括号匹配；
2. **队列**：模拟时间片轮转调度（Round Robin）；
3. **BFS**：从距离场扩展到真正的最短网格路径恢复。

请直接修改下面的 `TODO`，运行后面的测试单元。为了方便在 VS Code 里使用，未完成的练习测试会报告失败，而不会导致整个 Notebook 中断。

### 练习 1：实现括号匹配

写 `is_balanced(text)`：只检查 `()[]{}` 六种括号，其余字符忽略。

例子：

- `"a*(b+[c])"` → `True`
- `"([)]"` → `False`
- `"(()"` → `False`
- `"abc"` → `True`

**限制**：用 `list` 当栈，别直接调用上面讲解代码中的 `balanced_brackets`。

In [15]:
def is_balanced(text: str) -> bool:
    # TODO：创建匹配规则和空栈；扫描字符串
    # 遇到左括号入栈，遇到右括号检查匹配
    return None

### 练习 2：轮转调度（Round Robin）

假设有三个任务 `[("A", 2), ("B", 1), ("C", 3)]`，任务的第二项表示总共要运行多少个时间片。我们每次只允许**队头任务运行一个时间片**：

- 如果任务还有剩余时间，将它重新放回队尾；
- 如果已经完成，则不再重新入队。

输出应该是：

```text
['A', 'B', 'C', 'A', 'C', 'C']
```

编写 `round_robin(jobs)`，返回每个时间片里执行的任务名列表。假设时长均为正整数；不要修改输入参数。**提示**：`deque` 和 `popleft()`、`append()`。

In [16]:
def round_robin(jobs):
    # jobs: [(name, positive_integer_duration), ...]
    # TODO：把任务放入 deque，取队头运行一个时间片
    # 若任务未完成，放回队尾；记录实际执行顺序
    return None

### 练习 3：二维网格上的最短路径

编写 `shortest_grid_path(grid, start, goal)`。`grid` 是由 `'.'` 和 `'#'` 组成的等长字符串列表；如果有路，返回一个单元格坐标列表：

```python
[start, ..., goal]
```

没有路、起终点落在障碍物上时返回 `None`。可以复用之前的 `neighbors4()`。

**关键新知识：前驱映射 `parent`**。第一次发现新节点 `nxt` 时记录

```python
parent[nxt] = cell
```

最后从 `goal` 沿 `parent` 一步步回溯到 `start`，再反转列表。这样得到的是**实际路径**，而不只是最短步数。

可以用你刚学到的栈思路理解回溯。注意，在无权网格上 BFS 才能保证这条路径的步数最少。

In [17]:
def shortest_grid_path(grid, start, goal):
    # TODO：
    # 1) 检查起终点是否合法、是否被障碍物占据
    # 2) queue = deque([start]); parent = {start: None}
    # 3) 用 BFS 逐步发现邻居，并记录 parent[nxt] = cell
    # 4) 如果 goal 不可达返回 None，否则沿 parent 回溯并反转
    return None

### 8.4 一键自动检查

测试更关注**行为是否正确**，而不是要求你采用某一种完全一样的代码写法。如果提示失败，请先检查 `TODO` 是否已经替换，再检查返回类型和边界情况。

In [18]:
def check(name, test_fn):
    try:
        test_fn()
        print(f"✅ {name}：通过")
    except Exception as exc:
        print(f"⬜ {name}：尚未通过 — {type(exc).__name__}: {exc}")


def test_ex1():
    samples = {
        "": True, "abc": True, "a*(b+[c])": True,
        "([)]": False, "(()": False, ")": False, "{[]}": True,
    }
    for s, expected in samples.items():
        assert is_balanced(s) is expected, f"输入 {s!r} 应得到 {expected}"


def test_ex2():
    jobs = [("A", 2), ("B", 1), ("C", 3)]
    snapshot = list(jobs)
    assert round_robin(jobs) == ["A", "B", "C", "A", "C", "C"]
    assert jobs == snapshot, "不应修改输入 jobs"
    assert round_robin([]) == []
    assert round_robin([("X", 1)]) == ["X"]


def test_ex3():
    g = ["...", ".#.", "..."]
    path = shortest_grid_path(g, (0, 0), (2, 2))
    assert isinstance(path, list), "有路时应返回 list"
    assert path[0] == (0, 0) and path[-1] == (2, 2)
    assert len(path) == 5, "最短路径包含 5 个点 / 4 步"
    for a, b in zip(path, path[1:]):
        assert abs(a[0]-b[0]) + abs(a[1]-b[1]) == 1, "只能四邻域移动"
        assert g[b[0]][b[1]] != "#", "不能穿过障碍"
    assert shortest_grid_path([".#", "#."], (0, 0), (1, 1)) is None
    assert shortest_grid_path(g, (1, 1), (2, 2)) is None
    assert shortest_grid_path(g, (0, 0), (0, 0)) == [(0, 0)]


check("练习 1 / 栈", test_ex1)
check("练习 2 / 队列", test_ex2)
check("练习 3 / BFS 网格路径", test_ex3)

⬜ 练习 1 / 栈：尚未通过 — AssertionError: 输入 '' 应得到 True
⬜ 练习 2 / 队列：尚未通过 — AssertionError: 
⬜ 练习 3 / BFS 网格路径：尚未通过 — AssertionError: 有路时应返回 list


## 9. 参考答案（建议独立完成后运行）

下面的实现使用 `_solution` 后缀，不会覆盖你在练习中定义的函数。可以逐行对比实现，并研究三个细节：

1. 当栈为空时是否检查；
2. 重新入队时是否保留**剩余**时间；
3. BFS 的 `parent` 为什么在**第一次发现**节点时就记录。

In [19]:
def is_balanced_solution(text: str) -> bool:
    matching = {')': '(', ']': '[', '}': '{'}
    openers = set(matching.values())
    stack = []
    for ch in text:
        if ch in openers:
            stack.append(ch)
        elif ch in matching:
            if not stack or stack.pop() != matching[ch]:
                return False
    return not stack


def round_robin_solution(jobs):
    q = deque((name, duration) for name, duration in jobs)
    schedule = []
    while q:
        name, remaining = q.popleft()
        schedule.append(name)
        remaining -= 1
        if remaining > 0:
            q.append((name, remaining))
    return schedule


def shortest_grid_path_solution(grid, start, goal):
    if not grid or not grid[0]:
        return None
    ny, nx = len(grid), len(grid[0])
    for j, i in (start, goal):
        if not (0 <= j < ny and 0 <= i < nx):
            return None
        if grid[j][i] == '#':
            return None

    q = deque([start])
    parent = {start: None}

    while q:
        cell = q.popleft()
        if cell == goal:
            break
        for nxt in neighbors4(grid, cell):
            if nxt not in parent:
                parent[nxt] = cell
                q.append(nxt)

    if goal not in parent:
        return None

    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()
    return path


print("练习1答案:", is_balanced_solution("[a+(b*c)]"))
print("练习2答案:", round_robin_solution([("A", 2), ("B", 1), ("C", 3)]))
print("练习3答案:", shortest_grid_path_solution(mesh, start, goal))

练习1答案: True
练习2答案: ['A', 'B', 'C', 'A', 'C', 'C']
练习3答案: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (6, 6), (6, 7), (6, 8), (6, 9)]


### 9.1 将网格最短路径打印出来

用 `S` 标起点，`G` 标终点，用 `*` 表示经过的可通行网格。这个小工具也可以检查自己写的 `shortest_grid_path()` 是否正确：把 `path = ..._solution(...)` 改成你的函数即可。

In [20]:
def show_path(grid, start, goal, path):
    image = [list(row) for row in grid]
    if path is not None:
        for j, i in path:
            image[j][i] = '*'
    sj, si = start
    gj, gi = goal
    image[sj][si] = 'S'
    image[gj][gi] = 'G'
    for row in image:
        print(''.join(row))


path = shortest_grid_path_solution(mesh, start, goal)
show_path(mesh, start, goal, path)
if path is not None:
    print("路径步数:", len(path) - 1)

S.........
*####...#.
*#......#.
*#..####..
*#........
*..###....
*********G
路径步数: 15


## 10. 综合思考：把数据结构放回 CFD / JAX / 强化学习

考虑你希望搭建的“CFD → 代理模型 → 强化学习”流程：

| 需要解决的任务 | 适合考虑的数据结构 | 原因 |
|:--|:--|:--|
| 保存二维速度场 $u[j,i],v[j,i]$ | NumPy/JAX 数组 | 同类型密集数值与批量运算 |
| 检查障碍物是否把流体区域分成多块 | 队列/栈 + `set` | 逐单元遍历连通性 |
| 管理待执行 CFD 仿真任务 | 队列 | 按顺序取任务 |
| 保存最近 100 次迭代的残差 | `deque(maxlen=100)` | 自动丢弃最旧历史 |
| 用已训练代理模型批量推断流场 | JAX 数组 | GPU 上的张量计算 |
| 保存大量可随机采样的 RL 经验 | 数组或专用缓冲区 | 支持大批量索引、采样与传输 |

**不要把“算法使用了队列”误读为“GPU 上所有计算都应该用队列”**。队列帮助你处理离散任务的组织顺序；JAX/NumPy 数组帮助你高效执行大规模数值运算。这是两个不同层次的设计决策。

## 11. 自测与选做

不看前文，试着回答：

1. 栈和队列分别保证什么顺序？
2. `list.pop()`、`list.pop(0)` 和 `deque.popleft()` 的典型时间复杂度有何区别？
3. 为什么 BFS 使用队列，而迭代 DFS 常用栈？
4. 为什么 BFS 不能无条件求解任意加权图的最短路径？
5. 网格尺寸为 `ny=1000, nx=1000` 时，四邻域遍历为什么通常按 $O(n_y n_x)$ 计？
6. `visited` / `parent` 为什么适合暂时用 `set` / `dict`，而二维浮点场仍应放在数组中？

**选做挑战**：

- 编写 `flood_fill(grid, start)`，返回起点所在连通区域的全部单元格；再统计整个网格有多少个连通区域。
- 对同一网格，分别实现栈版 DFS 与队列版 BFS，比较找到目标前访问了多少节点；注意 **访问顺序与是否最短**是不同问题。
- 更进一步，把 `parent` 存成形状 `(ny, nx, 2)` 的整数 NumPy 数组，并用 `-1` 表示未发现的节点，观察与 `dict` 版本的差异。

**下一讲**：`dict`、`set` 与哈希表。你在本讲已经多次使用它们；下一讲会回答“为什么成员检查与查找通常如此快？有什么代价？”